<a href="https://colab.research.google.com/github/zzzer0-wav/myDTA_2026/blob/main/ML/Feature_engineering_categorical_features_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature engineering. Categorical features: One-Hot, Ordinal

## Налаштування та дані

- квартири (регресія) — з числовими та категорійними ознаками й датою

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# ---------- Датасет 1: КВАРТИРИ (регресія) ----------
n = 1500
cities = np.random.choice(["Київ", "Львів", "Харків", "Одеса"], n, p=[.4, .2, .2, .2])
city_premium = pd.Series({"Київ": 60, "Львів": 25, "Харків": 10, "Одеса": 20})

condition = np.random.choice(["аварійний", "житловий", "хороший", "євроремонт"],
                             n, p=[.1, .4, .35, .15])
cond_bonus = pd.Series({"аварійний": -20, "житловий": 0, "хороший": 15, "євроремонт": 40})

area    = np.random.normal(60, 20, n).clip(20, 140)
rooms   = np.clip(np.round(area / 25 + np.random.normal(0, .6, n)), 1, 5).astype(int)
floor   = np.random.randint(1, 25, n)
dist_km = np.random.exponential(5, n).clip(.3, 25)
listing_date = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 540, n), unit="D")

price = (40 + area*1.8 + rooms*5 + floor*.4 - dist_km*3
         + city_premium[cities].values + cond_bonus[condition].values
         + np.random.normal(0, 12, n)).clip(20, None)

apt = pd.DataFrame({
    "area": area.round(1), "rooms": rooms, "floor": floor,
    "dist_km": dist_km.round(1), "city": cities, "condition": condition,
    "listing_date": listing_date, "price": price.round(1),
})


print("Квартири:", apt.shape)
apt.head()

Квартири: (1500, 8)


,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


## Feature Engineering — створення нових ознак

In [21]:
df = apt.copy()
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


In [22]:
# Співвідношення
df['area_per_room'] = (df.area / df.rooms).round(1)
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price,area_per_room
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7,27.1
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3,24.1
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1,18.4
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1,32.7
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1,28.1


In [23]:
# Взаємодія
df['area_x_dist'] = (df.area * df.dist_km).round(1)
df[['area', 'dist_km', 'area_x_dist']].head()

,area,dist_km,area_x_dist
0,81.2,4.4,357.3
1,72.3,4.0,289.2
2,73.7,1.2,88.4
3,32.7,2.8,91.6
4,84.2,5.3,446.3


In [24]:
# Бінарна ознака: True(1) & False(0)
df['is_central'] = (df.dist_km < 3).astype(int)
df[['dist_km','is_central']].head()

,dist_km,is_central
0,4.4,0
1,4.0,0
2,1.2,1
3,2.8,1
4,5.3,0


In [25]:
# Бін - поділ на групи
df['floor_group'] = pd.cut(
    df.floor,
    bins=[0, 2, 9, 100],
    labels=['low', 'middle', 'high']
)

df[['floor', 'floor_group']].head()

,floor,floor_group
0,20,high
1,10,high
2,23,high
3,9,middle
4,2,low


In [27]:
# date
df['list_month'] = df.listing_date.dt.month
df['list_weekday'] = df.listing_date.dt.weekday
df['is_weekend'] = (df['list_weekday'] >= 5).astype(int)

df[['listing_date', 'list_month', 'list_weekday', 'is_weekend']].head()

,listing_date,list_month,list_weekday,is_weekend
0,2024-04-23,4,1,0
1,2025-01-31,1,4,0
2,2024-06-22,6,5,1
3,2024-09-11,9,2,0
4,2025-04-10,4,3,0


In [28]:
df.columns

Index(['area', 'rooms', 'floor', 'dist_km', 'city', 'condition', 'listing_date', 'price', 'area_per_room', 'area_x_dist', 'is_central',
       'floor_group', 'list_month', 'list_weekday', 'is_weekend'],
      dtype='object')

In [29]:
df[['area', 'rooms', 'area_per_room', 'dist_km', 'is_central', 'floor', 'floor_group', 'list_month', 'is_weekend']]

,area,rooms,area_per_room,dist_km,is_central,floor,floor_group,list_month,is_weekend
0,81.2,3,27.1,4.4,0,20,high,4,0
1,72.3,3,24.1,4.0,0,10,high,1,0
2,73.7,4,18.4,1.2,1,23,high,6,1
3,32.7,1,32.7,2.8,1,9,middle,9,0
4,84.2,3,28.1,5.3,0,2,low,4,0
...,...,...,...,...,...,...,...,...,...
1495,76.6,4,19.2,9.4,0,19,high,7,0
1496,81.8,3,27.3,6.6,0,14,high,7,1
1497,50.7,1,50.7,4.7,0,22,high,10,0
1498,57.0,1,57.0,8.6,0,13,high,3,0


In [42]:
check = df[['area', 'rooms', 'area_per_room', 'area_x_dist', 'dist_km', 'is_central', 'floor', 'price']].corr()['price']
check.drop('price').round(3).sort_values(key=abs, ascending=False)

,price
area,0.779
rooms,0.645
dist_km,-0.255
is_central,0.181
area_per_room,-0.052
area_x_dist,0.013
floor,0.010


Categorical features: One-Hot (номінальний), Ordinal (порядкові)

In [41]:
dummies = pd.get_dummies(df[['city', 'condition']], dtype=int)
print(f'Було 2 стовпчиків, стало - {dummies.shape[1]}, числових прапорців')
dummies.head()

Було 2 стовпчиків, стало - 8, числових прапорців


,city_Київ,city_Львів,city_Одеса,city_Харків,condition_аварійний,condition_житловий,condition_хороший,condition_євроремонт
0,1,0,0,0,0,0,1,0
1,0,0,1,0,0,1,0,0
2,0,0,0,1,1,0,0,0
3,0,1,0,0,0,1,0,0
4,1,0,0,0,0,1,0,0


In [36]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded = ohe.fit_transform(df[['city']])
print(list(ohe.categories_[0]))
pd.DataFrame(encoded[:5], columns=ohe.get_feature_names_out(['city'])).astype(int)

['Київ', 'Львів', 'Одеса', 'Харків']


,city_Київ,city_Львів,city_Одеса,city_Харків
0,1,0,0,0
1,0,0,1,0
2,0,0,0,1
3,0,1,0,0
4,1,0,0,0


### Ordinal (порядкові)

In [44]:
df['condition'].unique()

array(['хороший', 'житловий', 'аварійний', 'євроремонт'], dtype=object)

In [50]:
from sklearn.preprocessing import OrdinalEncoder

order = [['аварійний', 'житловий', 'хороший', 'євроремонт']]
oe = OrdinalEncoder(categories=order)

apt_demo = apt[['condition']].copy()
apt_demo['condition_code'] = oe.fit_transform(apt[['condition']]).astype(int)

apt_demo.drop_duplicates().sort_values('condition_code')

,condition,condition_code
2,аварійний,0
1,житловий,1
0,хороший,2
12,євроремонт,3


In [51]:
df['condition_code'] = oe.fit_transform(apt[['condition']]).astype(int)

df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price,area_per_room,area_x_dist,is_central,floor_group,list_month,list_weekday,is_weekend,condition_code
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7,27.1,357.3,0,high,4,1,0,2
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3,24.1,289.2,0,high,1,4,0,1
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1,18.4,88.4,1,high,6,5,1,0
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1,32.7,91.6,1,middle,9,2,0,1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1,28.1,446.3,0,low,4,3,0,1
